# Step 4: Vector Search (Semantic Recommendations)

This notebook builds a **semantic search engine** over book descriptions. Users can describe what they want in natural language (e.g. *"A story about forgiveness"*) and get relevant books — even if those exact words never appear in the catalog.

**How it works:**
1. Convert each book description to a **vector embedding** (a numeric representation of meaning).
2. Store embeddings in a **vector database** (ChromaDB).
3. At query time, embed the user's text and find the nearest book vectors (**similarity search**).

**Pipeline position:**
```
books_cleaned.csv  →  [this notebook]  →  Chroma index  →  Gradio dashboard
```

## Load data and prepare documents

Each line in `tagged_description.txt` starts with the book's ISBN so we can link search results back to metadata.

In [ ]:
# Load books and export tagged descriptions for embedding.
import pandas as pd
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

books = pd.read_csv("datasets/books_cleaned.csv")

with open("datasets/tagged_description.txt", "w", encoding="utf-8") as f:
    for desc in books["tagged_description"].astype(str):
        f.write(desc + "\n")

with open("datasets/tagged_description.txt", encoding="utf-8") as f:
    documents = [Document(page_content=line.strip()) for line in f if line.strip()]

print(f"Loaded {len(documents):,} book documents")

## Build the vector index

We use **HuggingFace sentence-transformers** (`all-MiniLM-L6-v2`) to embed text locally — no API key required. ChromaDB stores vectors and supports fast nearest-neighbor search.

> Alternative: OpenAI embeddings (`text-embedding-3-large`) often give higher quality but need an API key.

In [ ]:
# Embed documents and store in ChromaDB.
model_name = "sentence-transformers/all-MiniLM-L6-v2"
embedding_model = HuggingFaceEmbeddings(model_name=model_name)

db_books = Chroma.from_documents(documents, embedding=embedding_model)
print("Vector index ready.")

## Query the index

`similarity_search` returns the most semantically similar book descriptions. We parse the ISBN from the tagged text and look up full book details.

In [ ]:
# Retrieve book recommendations for a natural-language query.
def retrieve_semantic_recommendations(query: str, top_k: int = 10) -> pd.DataFrame:
    recs = db_books.similarity_search(query, k=50)
    books_list = [int(rec.page_content.strip('"').split()[0]) for rec in recs]
    return books[books["isbn13"].isin(books_list)].head(top_k)

retrieve_semantic_recommendations("A book to teach children about nature")